# Grok-GPU-T4x2-Smoke

> **Domain:** GPU · **Task:** dual-T4 smoke (GEMM + DataParallel CNN)
>
> Confirm Kaggle **T4×2** is visible, benchmark both devices, train a tiny CNN
> with `DataParallel`, and write artifacts under `/kaggle/working`.

## Naming
| Field | Value |
|---|---|
| Title | `Grok-GPU-T4x2-Smoke` |
| Slug | `grok-gpu-t4x2-smoke` |
| Accelerator | `NvidiaTeslaT4` (2×T4 on Kaggle) |


In [ ]:
# -*- dual-T4: do NOT pin CUDA_VISIBLE_DEVICES -*-
import os
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
print("CUDA_VISIBLE_DEVICES unset -> use all visible GPUs")


In [ ]:
import json, time, platform, traceback
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

OUT = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)

print("torch", torch.__version__)
print("python", platform.python_version())
print("cuda_available", torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA not available — enable GPU (NvidiaTeslaT4)"

n = torch.cuda.device_count()
print("device_count", n)
devices = []
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    info = {
        "index": i,
        "name": torch.cuda.get_device_name(i),
        "capability": list(torch.cuda.get_device_capability(i)),
        "mem_gb": round(p.total_memory / 1e9, 2),
    }
    devices.append(info)
    print(f"  [{i}] {info['name']}  {info['mem_gb']}GB  cap={info['capability']}")

# Prefer dual T4; still succeed on 1×T4 with a warning so re-runs aren't blocked
if n < 2:
    print("WARNING: expected 2×T4, got", n, "— continuing on available GPU(s)")
else:
    print("OK: dual GPU visible")

DEVICE = torch.device("cuda:0")
print("primary", DEVICE)


In [ ]:
# Per-device FP32 GEMM micro-benchmark
def gemm_bench(device_idx: int, N: int = 4096, iters: int = 10):
    dev = torch.device(f"cuda:{device_idx}")
    a = torch.randn(N, N, device=dev)
    b = torch.randn(N, N, device=dev)
    for _ in range(3):
        c = a @ b
    torch.cuda.synchronize(dev)
    t0 = time.perf_counter()
    for _ in range(iters):
        c = a @ b
    torch.cuda.synchronize(dev)
    elapsed = time.perf_counter() - t0
    flops = 2 * (N ** 3) * iters
    tflops = flops / elapsed / 1e12
    return {"device": device_idx, "n": N, "iters": iters, "seconds": elapsed, "tflops_fp32": tflops}

gemm_results = []
for i in range(torch.cuda.device_count()):
    r = gemm_bench(i)
    gemm_results.append(r)
    print(f"GEMM cuda:{i} {r['n']}x{r['n']} x{r['iters']}: {r['seconds']:.3f}s, ~{r['tflops_fp32']:.2f} TFLOPS")


In [ ]:
# Tiny CNN + optional DataParallel across visible T4s
class TinyCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, n_classes),
        )
    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

n_train, n_val = 4096, 512
x_train = torch.randn(n_train, 3, 32, 32)
y_train = torch.randint(0, 10, (n_train,))
x_val = torch.randn(n_val, 3, 32, 32)
y_val = torch.randint(0, 10, (n_val,))

# Larger batch when dual GPU is available
n_gpu = torch.cuda.device_count()
batch_size = 128 * max(n_gpu, 1)
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=256, shuffle=False, num_workers=0)

model = TinyCNN().to(DEVICE)
used_dp = False
if n_gpu > 1:
    model = nn.DataParallel(model)
    used_dp = True
    print(f"DataParallel on {n_gpu} GPUs, batch_size={batch_size}")
else:
    print(f"single GPU, batch_size={batch_size}")

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

history = []
epochs = 5
t0 = time.perf_counter()
for epoch in range(1, epochs + 1):
    model.train()
    total_loss, n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        opt.step()
        total_loss += loss.item() * xb.size(0)
        n += xb.size(0)
    train_loss = total_loss / n

    model.eval()
    correct, n = 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb).argmax(dim=1)
            correct += (pred == yb).sum().item()
            n += yb.size(0)
    val_acc = correct / n
    history.append({"epoch": epoch, "train_loss": train_loss, "val_acc": val_acc})
    print(f"epoch {epoch}/{epochs}  train_loss={train_loss:.4f}  val_acc={val_acc:.3f}")

train_seconds = time.perf_counter() - t0
print(f"train_time_s={train_seconds:.2f}")


In [ ]:
# Persist summary + checkpoint
raw = model.module if isinstance(model, nn.DataParallel) else model
params = sum(p.numel() for p in raw.parameters())

out = {
    "notebook": "Grok-GPU-T4x2-Smoke",
    "domain": "GPU",
    "task": "T4x2-Smoke",
    "cuda_available": True,
    "device_count": torch.cuda.device_count(),
    "devices": devices,
    "data_parallel": used_dp,
    "batch_size": batch_size,
    "gemm": gemm_results,
    "train_seconds": train_seconds,
    "history": history,
    "params": params,
    "torch": torch.__version__,
    "python": platform.python_version(),
    "status": "ok",
}

path = OUT / "grok_gpu_t4x2_smoke_results.json"
path.write_text(json.dumps(out, indent=2))
print("wrote", path)
print(json.dumps(out, indent=2))

ckpt = OUT / "tiny_cnn_dp.pt"
torch.save({"model": raw.state_dict(), "history": history, "meta": out}, ckpt)
print("wrote", ckpt)

# Hard fail only if CUDA path completely broken
assert out["status"] == "ok"
assert len(history) == 5
assert history[-1]["train_loss"] < history[0]["train_loss"] or history[-1]["val_acc"] > 0.05
print("SMOKE PASS")


## Checklist
- [x] CUDA devices printed (expect 2× Tesla T4)
- [x] GEMM TFLOPS on each device
- [x] DataParallel CNN epochs complete
- [x] Results JSON + checkpoint under `/kaggle/working`
